# CAITANYA real-model experiment
Select a GPU (L4 preferred; T4 supported). This notebook never substitutes mock inference. Drive authorization is required. The viewer uses Colab’s authenticated port proxy; do not create a public tunnel. Each run is bounded. Re-running launch refuses to replace a running process. Preserve run IDs to recover after runtime restart.

In [ ]:
from google.colab import drive
from pathlib import Path
import subprocess, sys, json, datetime, os, time
import torch
assert torch.cuda.is_available(), 'A CUDA GPU is required; no mock fallback'
print('GPU:', torch.cuda.get_device_name(0))
drive.mount('/content/drive')
STORE=Path('/content/drive/MyDrive/CAITANYA'); STORE.mkdir(exist_ok=True)
SOURCE=Path('/content/AI-World')
if not SOURCE.exists(): subprocess.run(['git','clone','https://github.com/CoderVLSI/AI-World.git',str(SOURCE)],check=True)
subprocess.run(['git','fetch','origin','main'],cwd=SOURCE,check=True)
subprocess.run(['git','checkout','--detach','origin/main'],cwd=SOURCE,check=True)
SHA=subprocess.check_output(['git','rev-parse','HEAD'],cwd=SOURCE,text=True).strip()
print('SOURCE COMMIT:',SHA)
subprocess.run([sys.executable,'-m','pip','install','-e',str(SOURCE)+'[gpu,test]'],check=True)
subprocess.run([sys.executable,'-m','pytest','-q'],cwd=SOURCE,check=True)


In [ ]:
# Keep this ID after runtime restart to recover the same world.
subprocess.run([sys.executable, str(SOURCE/'scripts/storage_preflight.py'), str(STORE)], check=True)
RUN_ID='five-'+datetime.datetime.now(datetime.timezone.utc).strftime('%Y%m%dT%H%M%SZ')
POPULATION=5  # Attempt 50 only after a verified five-agent report.
ROOT_MODE='observe'  # off: baseline; observe: records only; crisis: bounded intervention.
MAX_TICKS=30 if POPULATION==5 else 60
MAX_SECONDS=180 if POPULATION==5 else 300
RUN=STORE/RUN_ID
REPORT=RUN/'summary'
PORT=8003
from huggingface_hub import model_info
REVISION=model_info('LiquidAI/LFM3-1.7B').sha
# Recovery must use the checkpoint's original revision and stored root mode.
images=sorted(RUN.glob('checkpoint-*.json')) if RUN.exists() else []
if images:
    REVISION=json.loads(images[-1].read_text())['body']['state']['manifest']['model_revision']
if 'world_process' in globals() and world_process.poll() is None:
    raise RuntimeError('Stop and checkpoint the existing process before launching another.')
RUN.mkdir(exist_ok=True)
cmd=[sys.executable,'-m','caitanya.server','--backend','lfm3','--revision',REVISION,
     '--root-mode',ROOT_MODE,'--population',str(POPULATION),'--width','256','--depth','256',
     '--run-dir',str(RUN),'--report-dir',str(REPORT),'--port',str(PORT),
     '--max-ticks',str(MAX_TICKS),'--max-seconds',str(MAX_SECONDS)]
if images: cmd+=['--recover',str(RUN)]
(RUN/'launch.json').write_text(json.dumps({'commit':SHA,'command':cmd,'model_revision':REVISION},indent=2))
log=(RUN/'process.log').open('a')
world_process=subprocess.Popen(cmd,cwd=SOURCE,stdout=log,stderr=subprocess.STDOUT)
print('PID',world_process.pid,'RUN',RUN_ID,'MODEL',REVISION)
print('Run the next cell after the server has loaded. Startup errors remain in process.log.')


In [ ]:
import requests
BASE=f'http://127.0.0.1:{PORT}'
print((RUN/'process.log').read_text()[-2500:])
health=requests.get(BASE+'/api/v1/health',timeout=10).json()
assert health['mode']=='TransformersLFM3Backend', health
snapshot=requests.get(BASE+'/api/v1/world/snapshot',timeout=10).json()
assert snapshot['manifest']['model_id']=='LiquidAI/LFM3-1.7B'
assert requests.post(BASE+'/api/v1/world/snapshot',json={},timeout=10).status_code==405
print('VERIFIED BACKEND',health,'METRICS',requests.get(BASE+'/api/v1/metrics',timeout=10).json())
from google.colab import output
output.serve_kernel_port_as_window(PORT)


In [ ]:
# The worker stops at the budget; the read-only viewer remains available.
if (REPORT/'evidence.json').exists():
    evidence=json.loads((REPORT/'evidence.json').read_text())
    print('Verified real identities:',evidence['verified_real_agent_ids'])
    print('Inference:',evidence['inference'])
    print('Recovery:',evidence['metadata']['acceptance'])
    print((REPORT/'REPORT.md').read_text()[:6000])
else:
    print('Run still in progress or startup failed. Inspect metrics and process.log.')
# Graceful stop when finished: world_process.terminate(); world_process.wait(timeout=90)
# After runtime restart: rerun setup; set RUN_ID to the prior ID in launch cell.
# The CLI recovers committed history, preserving stored root mode and model revision.
# Download/review only summary/ files for GitHub. Never upload raw journals or checkpoints.
